# FloodShift-04E — FINAL TerraMind-RIFT High-Standard Run

**Purpose:** one final, performance-first, scientifically controlled FloodShift segmentation run.

### Final architecture in this notebook
1. **TerraMind-v1-base** pretrained Earth-observation backbone.
2. Native **Sentinel-2 L1C + Sentinel-1 GRD** multimodal token encoder.
3. Frozen project bands only:
   - SAR: `VV, VH`
   - Optical: `B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12`
4. TerraMind intermediate transformer layers: **[2, 5, 8, 11]**.
5. The TerraMind `concat` merge keeps modality embeddings separable in the feature dimension.
6. A **zero-initialized deep Reliability-Conditioned Cross Attention (RIFT)** block lets:
   - SAR borrow optical context mainly where `P(clear)` is high.
   - Optical borrow SAR context mainly where `P(clear)` is low.
7. Four **P(clear)-conditioned spatial routers** fuse SAR/optical features.
8. Multiscale reliability-routed U-Net/ConvNeXt-style decoder.
9. Main objective:
   - 45% class-balanced CE
   - 35% Lovasz-Softmax
   - 20% temporary-flood focal Tversky
   - small auxiliary CE
10. **Conservative transfer schedule**:
    - Epochs 1–5: TerraMind backbone frozen.
    - Then: unfreeze **only the final two TerraMind transformer blocks** at a very small LR.
    - Never full-unfreeze the backbone in this run.
11. Top-3 validation checkpoints are retained. A validation-only weight-averaged checkpoint is tested against the best single checkpoint.
12. Four-way TTA is enabled only if it improves validation mIoU by a predeclared margin.
13. The held-out test set is automatically evaluated **only if the predeclared validation promotion gate passes**.

### Required Kaggle inputs
Attach **only**:
- **Input A — Notebook 01 / raw audited Sen1Floods11 source**
  - must make the original `S1Hand`, `S2Hand`, `LabelHand`, and `JRCWaterHand` files reachable under `/kaggle/input`.
- **Input B — final saved output of Notebook 03**
  - `floodshift_preprocessing_manifest_v1.csv`
  - `floodshift_preprocessing_GLOBAL_LOCK_v1.json`
  - `optical_10band_train_normalization_exact.csv`
  - `train_raw_sar_scene_statistics.csv`
  - all frozen `optical_reliability` arrays referenced by the manifest.

**NOT required:** V1 checkpoint, V2 checkpoint, V3 checkpoint, Notebook 04A, or Notebook 04D.

### Kaggle settings
- Accelerator: **GPU T4**
- Use **one GPU only**.
- Internet: **ON** for TerraTorch / TerraMind pretrained weights.
- **Use a fresh Kaggle session. Do not run the old TerraTorch 1.2.x install cell.**
- Run cells strictly from top to bottom.
- Do not alter the geographic split, target mapping, preprocessing, or test gate.

In [1]:
import numpy as np
import pandas as pd
import scipy
import torch

print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy :", scipy.__version__)
print("Torch :", torch.__version__)

import terratorch

print("TerraTorch:", getattr(terratorch, "__version__", "unknown"))
print("\n✅ ENVIRONMENT HEALTHY")

NumPy : 1.26.4
Pandas: 2.3.3
SciPy : 1.16.3
Torch : 2.10.0+cu128


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


TerraTorch: unknown

✅ ENVIRONMENT HEALTHY


In [2]:
import terratorch

print(
    "TerraTorch:",
    getattr(terratorch, "__version__", "unknown")
)

print("✅ TERRATORCH IMPORT SUCCESS")

TerraTorch: unknown
✅ TERRATORCH IMPORT SUCCESS


In [ ]:
# ============================================================
# CELL: P7-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
#
# WHY 1.1.1?
# Kaggle's current Python 3.12 image ships with NumPy 2.0.x.
# TerraTorch 1.2.x requires NumPy >= 2.2 and can destabilize the
# precompiled Kaggle SciPy/NumPy stack when installed in-place.
#
# TerraTorch 1.1.1 already includes TerraMind v1 support,
# S2L1C + S1GRD multimodal inputs, concat merging, band subsets,
# and BACKBONE_REGISTRY, without forcing NumPy >= 2.2.
# ============================================================

import sys
import subprocess

print("Installing Kaggle-compatible TerraTorch 1.1.1...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-cache-dir",
    "terratorch==1.1.1",
])

print("P7-C00 COMPLETE ✅")
print("IMPORTANT: Do NOT restart the session after this cell.")

In [ ]:
import urllib.request

url = "https://pypi.org/simple/terratorch/"

try:
    response = urllib.request.urlopen(url, timeout=10)
    print("✅ INTERNET WORKING")
    print("Status:", response.status)
except Exception as e:
    print("❌ INTERNET NOT WORKING")
    print(type(e).__name__, ":", e)

In [4]:
# ============================================================
# CELL: P7-C01 — ENVIRONMENT + REPRODUCIBILITY
# SINGLE-GPU ONLY
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
import gc
import re
import json
import time
import math
import random
import copy
from pathlib import Path

import numpy as np
import scipy
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

AMP_ENABLED = torch.cuda.is_available()

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_FINAL_TerraMind_RIFT")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
HISTORY_ROOT = OUTPUT_ROOT / "history"
METRIC_ROOT = OUTPUT_ROOT / "metrics"
CALIBRATION_ROOT = OUTPUT_ROOT / "calibration"

for p in [
    OUTPUT_ROOT,
    CHECKPOINT_ROOT,
    HISTORY_ROOT,
    METRIC_ROOT,
    CALIBRATION_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("P7-C01 — FINAL TERRAMIND-RIFT ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("SciPy       :", scipy.__version__)
print("PyTorch     :", torch.__version__)
print("TerraTorch  :", getattr(terratorch, "__version__", "unknown"))
print("Device      :", DEVICE)
print("Visible GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))

print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)

if torch.cuda.device_count() > 1:
    print(
        f"ℹ️ {torch.cuda.device_count()} GPUs visible, "
        "but FloodShift will use cuda:0 only."
    )

assert DEVICE == torch.device("cuda:0")

P7-C01 — FINAL TERRAMIND-RIFT ENVIRONMENT
Python      : 3.12.13
NumPy       : 1.26.4
SciPy       : 1.16.3
PyTorch     : 2.10.0+cu128
TerraTorch  : unknown
Device      : cuda:0
Visible GPUs: 2
GPU         : Tesla T4
AMP         : True
Output root : /kaggle/working/FloodShift_FINAL_TerraMind_RIFT
ℹ️ 2 GPUs visible, but FloodShift will use cuda:0 only.


In [5]:
# ============================================================
# CELL: P7-C02 — LOCATE FROZEN NOTEBOOK-03 ARTIFACTS
# ============================================================

_file_cache = {}

def locate_by_saved_path(saved_path_or_name):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    cache_key = str(saved_path_or_name)

    if cache_key in _file_cache:
        return _file_cache[cache_key]

    matches = list(INPUT_ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not locate '{filename}'. "
            "Attach the audited raw-data input (Notebook 01/source data) "
            "and the final successful Notebook 03 output."
        )

    parent_hint = saved.parent.name.lower()

    hinted = [
        p for p in matches
        if parent_hint and parent_hint in str(p).lower()
    ]

    candidates = hinted if hinted else matches

    selected = sorted(
        candidates,
        key=lambda p: (len(str(p)), str(p))
    )[0]

    _file_cache[cache_key] = selected
    return selected


FINAL_MANIFEST_PATH = locate_by_saved_path(
    "floodshift_preprocessing_manifest_v1.csv"
)

GLOBAL_LOCK_PATH = locate_by_saved_path(
    "floodshift_preprocessing_GLOBAL_LOCK_v1.json"
)

OPTICAL_NORM_PATH = locate_by_saved_path(
    "optical_10band_train_normalization_exact.csv"
)

SAR_SCENE_STATS_PATH = locate_by_saved_path(
    "train_raw_sar_scene_statistics.csv"
)

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 112)
print("P7-C02 — FROZEN PREPROCESSING FOUND")
print("=" * 112)

for p in [
    FINAL_MANIFEST_PATH,
    GLOBAL_LOCK_PATH,
    OPTICAL_NORM_PATH,
    SAR_SCENE_STATS_PATH,
]:
    print(" ", p)

print("\nLock:", preprocessing_lock["status"])
print("P7-C02 COMPLETE ✅")

P7-C02 — FROZEN PREPROCESSING FOUND
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_manifest_v1.csv
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_GLOBAL_LOCK_v1.json
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/optical/optical_10band_train_normalization_exact.csv
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/train_raw_sar_scene_statistics.csv

Lock: PERMANENTLY_FROZEN
P7-C02 COMPLETE ✅


In [6]:
# ============================================================
# CELL: P7-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P7-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")

P7-C03 — DATA CONFIG
Train / Val / Test: 314 68 58
Optical bands      : ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
SAR mean [VV,VH]   : [-10.358468 -17.10712 ]
SAR std  [VV,VH]   : [4.161632  4.8134336]
Reliability        : frozen continuous P(clear)
Primary SAR input  : raw VV/VH


In [7]:
# ============================================================
# CELL: P7-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FloodShiftDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }

In [8]:
# ============================================================
# CELL: P7-C05 — TARGET-SEMANTICS AUDIT + CLASS WEIGHTS
#
# This cell audits the mapping; it does not change it.
# ============================================================

class_counts = np.zeros(
    NUM_CLASSES,
    dtype=np.int64,
)

ignored_count = 0

semantic_counts = {
    "valid_pixels": 0,
    "flood_positive": 0,
    "jrc_positive": 0,
    "flood1_jrc1": 0,
    "flood1_jrc0": 0,
    "flood0_jrc1": 0,
}

for position, (_, row) in enumerate(
    train_df.iterrows(),
    start=1,
):
    paths = resolve_scene_files(row)

    with rasterio.open(
        paths["flood"]
    ) as src:
        flood = src.read(1)

    with rasterio.open(
        paths["jrc"]
    ) as src:
        jrc = src.read(1)

    target = build_three_class_target(
        flood,
        jrc,
    )

    valid = flood != -1

    semantic_counts[
        "valid_pixels"
    ] += int(valid.sum())

    semantic_counts[
        "flood_positive"
    ] += int(
        (valid & (flood == 1)).sum()
    )

    semantic_counts[
        "jrc_positive"
    ] += int(
        (valid & (jrc == 1)).sum()
    )

    semantic_counts[
        "flood1_jrc1"
    ] += int(
        (
            valid
            & (flood == 1)
            & (jrc == 1)
        ).sum()
    )

    semantic_counts[
        "flood1_jrc0"
    ] += int(
        (
            valid
            & (flood == 1)
            & (jrc != 1)
        ).sum()
    )

    semantic_counts[
        "flood0_jrc1"
    ] += int(
        (
            valid
            & (flood == 0)
            & (jrc == 1)
        ).sum()
    )

    for c in range(NUM_CLASSES):
        class_counts[c] += int(
            (target == c).sum()
        )

    ignored_count += int(
        (
            target
            == IGNORE_INDEX
        ).sum()
    )

    if (
        position == 1
        or position % 75 == 0
        or position == len(train_df)
    ):
        print(
            f"[{position:03d}/"
            f"{len(train_df)}] "
            "target audit"
        )

valid_total = class_counts.sum()

frequencies = (
    class_counts
    / valid_total
)

# Less aggressive than full inverse-frequency weighting.
# This raises temporary-flood relative importance while
# avoiding an extreme permanent-water weight.
weights_np = (
    1.0
    / np.sqrt(
        np.maximum(
            frequencies,
            1e-12,
        )
    )
)

weights_np = (
    weights_np
    / weights_np.mean()
)

weights_np = np.clip(
    weights_np,
    0.25,
    2.00,
)

weights_np = (
    weights_np
    / weights_np.mean()
)

CLASS_WEIGHTS = torch.tensor(
    weights_np,
    dtype=torch.float32,
    device=DEVICE,
)

target_audit = pd.DataFrame({
    "class_id": np.arange(
        NUM_CLASSES
    ),
    "class_name": CLASS_NAMES,
    "pixel_count": class_counts,
    "frequency": frequencies,
    "sqrt_inverse_weight": weights_np,
})

print("\n" + "=" * 112)
print("P7-C05 — TRAIN TARGET DISTRIBUTION")
print("=" * 112)

print(
    target_audit.to_string(
        index=False
    )
)

print(
    "\nIgnored:",
    f"{ignored_count:,}",
)

print(
    "\nCurrent semantic overlap counts:"
)

for k, v in semantic_counts.items():
    print(
        f"  {k:18s}: "
        f"{v:,}"
    )

print(
    "\nIMPORTANT: "
    "class-1 still means ANY valid JRC==1 pixel. "
    "No target semantics were changed."
)

target_audit.to_csv(
    METRIC_ROOT
    / "final_train_target_distribution.csv",
    index=False,
)

with open(
    METRIC_ROOT
    / "target_semantics_audit.json",
    "w",
) as f:
    json.dump(
        semantic_counts,
        f,
        indent=2,
    )

assert (
    class_counts > 0
).all()

[001/314] target audit
[075/314] target audit
[150/314] target audit
[225/314] target audit
[300/314] target audit
[314/314] target audit

P7-C05 — TRAIN TARGET DISTRIBUTION
 class_id      class_name  pixel_count  frequency  sqrt_inverse_weight
        0       non_flood     64724902   0.892878             0.292547
        1 permanent_water      1826059   0.025190             1.741703
        2 temporary_flood      5939270   0.081932             0.965750

Ignored: 9,822,985

Current semantic overlap counts:
  valid_pixels      : 72,490,231
  flood_positive    : 7,710,692
  jrc_positive      : 1,826,059
  flood1_jrc1       : 1,771,422
  flood1_jrc0       : 5,939,270
  flood0_jrc1       : 54,637

IMPORTANT: class-1 still means ANY valid JRC==1 pixel. No target semantics were changed.


In [9]:
# ============================================================
# CELL: P7-C06 — DATALOADERS + ONE-BATCH CHECK
# ============================================================

# TerraMind-v1-base + custom decoder is performance-first.
# Batch 2 + accumulation 4 gives effective batch 8.
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 2

def seed_worker(worker_id):
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


train_dataset = FloodShiftDataset(
    train_df,
    train_mode=True,
)

val_dataset = FloodShiftDataset(
    val_df,
    train_mode=False,
)

test_dataset = FloodShiftDataset(
    test_df,
    train_mode=False,
)


def make_loaders(seed=SEED):
    g = torch.Generator()

    g.manual_seed(
        seed
    )

    common = {
        "num_workers": NUM_WORKERS,
        "pin_memory": torch.cuda.is_available(),
        "worker_init_fn": seed_worker,
        "persistent_workers": (
            NUM_WORKERS > 0
        ),
    }

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=g,
        **common,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
    )


(
    train_loader,
    val_loader,
    test_loader,
) = make_loaders()

batch = next(
    iter(train_loader)
)

print("=" * 112)
print("P7-C06 — MULTIMODAL BATCH CHECK")
print("=" * 112)
print("SAR        :", tuple(batch["sar"].shape))
print("Optical    :", tuple(batch["optical"].shape))
print("Reliability:", tuple(batch["reliability"].shape))
print("Target     :", tuple(batch["target"].shape))
print(
    "P(clear)   :",
    float(
        batch["reliability"].min()
    ),
    "to",
    float(
        batch["reliability"].max()
    ),
)

assert (
    batch["sar"].shape[1:]
    == (2, 256, 256)
)

assert (
    batch["optical"].shape[1:]
    == (10, 256, 256)
)

assert (
    batch["reliability"].shape[1:]
    == (1, 256, 256)
)

assert (
    batch["target"].shape[1:]
    == (256, 256)
)

assert torch.isfinite(
    batch["sar"]
).all()

assert torch.isfinite(
    batch["optical"]
).all()

assert torch.isfinite(
    batch["reliability"]
).all()

print("P7-C06 COMPLETE ✅")

P7-C06 — MULTIMODAL BATCH CHECK
SAR        : (2, 2, 256, 256)
Optical    : (2, 10, 256, 256)
Reliability: (2, 1, 256, 256)
Target     : (2, 256, 256)
P(clear)   : 0.000995635986328125 to 0.9970703125
P7-C06 COMPLETE ✅


## Architecture note

TerraMind's pretrained backbone receives the **same frozen project tensors** after a deterministic model-input adapter:

- frozen optical z-score → reconstruct reflectance → reconstruct L1C DN → TerraMind standardization,
- frozen SAR z-score → reconstruct dB → TerraMind S1GRD standardization.

This is **not a new preprocessing search**. It is a fixed affine input adapter needed to place the already-frozen inputs into the pretrained backbone's expected scale.

The model uses TerraMind's official 10-band subset support for `S2L1C` and all `VV/VH` S1GRD bands.

In [10]:
# ============================================================
# CELL: P7-C07 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=True,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits


print("Building TerraMind-v1-base pretrained backbone...")
print("First execution may download pretrained weights.")

model = FloodShiftTerraMindRIFT(
    num_classes=NUM_CLASSES
).to(DEVICE)

print("Model built ✅")

Building TerraMind-v1-base pretrained backbone...
First execution may download pretrained weights.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-base/resolve/main/TerraMind_v1_base.pt "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/ibm-esa-geospatial/TerraMind-1.0-base/xet-read-token/fb96c70d0a5f68dcc44030b89cbfd8ec3fb0c67a "HTTP/1.1 200 OK"


TerraMind_v1_base.pt:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

INFO:root:Loaded weights for BLUE in position 0 of patch embed
INFO:root:Loaded weights for GREEN in position 1 of patch embed
INFO:root:Loaded weights for RED in position 2 of patch embed
INFO:root:Loaded weights for RED_EDGE_1 in position 3 of patch embed
INFO:root:Loaded weights for RED_EDGE_2 in position 4 of patch embed
INFO:root:Loaded weights for RED_EDGE_3 in position 5 of patch embed
INFO:root:Loaded weights for NIR_BROAD in position 6 of patch embed
INFO:root:Loaded weights for NIR_NARROW in position 7 of patch embed
INFO:root:Loaded weights for SWIR_1 in position 8 of patch embed
INFO:root:Loaded weights for SWIR_2 in position 9 of patch embed


Model built ✅


In [11]:
# ============================================================
# TERRAMIND POSITION/TOKEN DIAGNOSTIC
# DO NOT TRAIN YET
# ============================================================

print("Encoder embedding modules:")

for name, module in model.backbone.encoder_embeddings.items():
    pos = getattr(module, "pos_emb", None)
    image_size = getattr(module, "image_size", None)

    print("\n", name)
    print("  image_size:", image_size)

    if pos is not None:
        print("  pos_emb shape:", tuple(pos.shape))

print("\nTesting actual 256x256 FloodShift input...")

with torch.no_grad():
    sar = batch["sar"][:1].to(DEVICE)
    optical = batch["optical"][:1].to(DEVICE)

    sar_tm, optical_tm = model.to_terramind_inputs(
        sar,
        optical
    )

    print("S1 input:", tuple(sar_tm.shape))
    print("S2 input:", tuple(optical_tm.shape))

    outputs = model.backbone({
        "S2L1C": optical_tm,
        "S1GRD": sar_tm,
    })

    print("Number of transformer outputs:", len(outputs))

    for idx in [2, 5, 8, 11]:
        print(
            f"Layer {idx}:",
            tuple(outputs[idx].shape)
        )

print("\n✅ BACKBONE FORWARD SUCCESS")

Encoder embedding modules:

 untok_sen2l1c@224
  image_size: (256, 256)
  pos_emb shape: (1, 256, 768)

 untok_sen1grd@224
  image_size: (256, 256)
  pos_emb shape: (1, 256, 768)

Testing actual 256x256 FloodShift input...
S1 input: (1, 2, 256, 256)
S2 input: (1, 10, 256, 256)
Number of transformer outputs: 12
Layer 2: (1, 256, 1536)
Layer 5: (1, 256, 1536)
Layer 8: (1, 256, 1536)
Layer 11: (1, 256, 1536)

✅ BACKBONE FORWARD SUCCESS


In [12]:
# ============================================================
# TERRAMIND PRETRAINED POSITION-EMBEDDING INTERPOLATION
# 14x14 pretrained grid -> 16x16 FloodShift grid
# ============================================================

import gc
import torch
import torch.nn.functional as F

print("Building temporary 224x224 reference backbone...")

reference_backbone = BACKBONE_REGISTRY.build(
    "terramind_v1_base",
    pretrained=True,
    img_size=224,
    modalities=[
        "S2L1C",
        "S1GRD",
    ],
    bands={
        "S2L1C": TM_OPTICAL_BAND_NAMES,
    },
    merge_method="concat",
)

print("\nReference positional embeddings:")

for name, module in reference_backbone.encoder_embeddings.items():
    pos = getattr(module, "pos_emb", None)

    if pos is not None:
        print(
            name,
            tuple(pos.shape)
        )

print("\nInterpolating pretrained positional embeddings...")

with torch.no_grad():

    for name, target_module in model.backbone.encoder_embeddings.items():

        if name not in reference_backbone.encoder_embeddings:
            print("⚠️ Missing reference module:", name)
            continue

        source_module = reference_backbone.encoder_embeddings[name]

        source_pos = getattr(
            source_module,
            "pos_emb",
            None,
        )

        target_pos = getattr(
            target_module,
            "pos_emb",
            None,
        )

        if source_pos is None or target_pos is None:
            continue

        print(
            f"\n{name}: "
            f"{tuple(source_pos.shape)} -> "
            f"{tuple(target_pos.shape)}"
        )

        source_tokens = source_pos.shape[1]
        target_tokens = target_pos.shape[1]
        embed_dim = source_pos.shape[2]

        source_side = int(
            source_tokens ** 0.5
        )

        target_side = int(
            target_tokens ** 0.5
        )

        assert (
            source_side * source_side
            == source_tokens
        )

        assert (
            target_side * target_side
            == target_tokens
        )

        source_grid = (
            source_pos
            .reshape(
                1,
                source_side,
                source_side,
                embed_dim,
            )
            .permute(
                0,
                3,
                1,
                2,
            )
            .float()
        )

        interpolated = F.interpolate(
            source_grid,
            size=(
                target_side,
                target_side,
            ),
            mode="bicubic",
            align_corners=False,
        )

        interpolated = (
            interpolated
            .permute(
                0,
                2,
                3,
                1,
            )
            .reshape(
                1,
                target_tokens,
                embed_dim,
            )
        )

        target_module.pos_emb.copy_(
            interpolated.to(
                device=target_pos.device,
                dtype=target_pos.dtype,
            )
        )

        print("✅ pretrained position embedding transferred")

# Temporary 224 model no longer needed.
del reference_backbone
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n============================================================")
print("✅ TERRAMIND POSITIONAL EMBEDDINGS FIXED")
print("Pretrained 14x14 grid interpolated to FloodShift 16x16 grid")
print("============================================================")

Building temporary 224x224 reference backbone...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-base/resolve/main/TerraMind_v1_base.pt "HTTP/1.1 302 Found"
INFO:root:Loaded weights for BLUE in position 0 of patch embed
INFO:root:Loaded weights for GREEN in position 1 of patch embed
INFO:root:Loaded weights for RED in position 2 of patch embed
INFO:root:Loaded weights for RED_EDGE_1 in position 3 of patch embed
INFO:root:Loaded weights for RED_EDGE_2 in position 4 of patch embed
INFO:root:Loaded weights for RED_EDGE_3 in position 5 of patch embed
INFO:root:Loaded weights for NIR_BROAD in position 6 of patch embed
INFO:root:Loaded weights for NIR_NARROW in position 7 of patch embed
INFO:root:Loaded weights for SWIR_1 in position 8 of patch embed
INFO:root:Loaded weights for SWIR_2 in position 9 of patch embed



Reference positional embeddings:
untok_sen2l1c@224 (1, 196, 768)
untok_sen1grd@224 (1, 196, 768)

Interpolating pretrained positional embeddings...

untok_sen2l1c@224: (1, 196, 768) -> (1, 256, 768)
✅ pretrained position embedding transferred

untok_sen1grd@224: (1, 196, 768) -> (1, 256, 768)
✅ pretrained position embedding transferred

✅ TERRAMIND POSITIONAL EMBEDDINGS FIXED
Pretrained 14x14 grid interpolated to FloodShift 16x16 grid


In [13]:
# ============================================================
# CELL: P7-C08 — SHAPE CHECK + CONSERVATIVE FREEZE POLICY
# ============================================================

def freeze_entire_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False


def unfreeze_last_backbone_blocks(
    model,
    last_n=2,
    depth=12,
):
    freeze_entire_backbone(
        model
    )

    min_block = (
        depth - last_n
    )

    matched = []

    for name, p in (
        model.backbone.named_parameters()
    ):
        # TerraMind core exposes encoder.<block index>.
        match = re.search(
            r"(?:^|\.)encoder\.(\d+)\.",
            name,
        )

        if match:
            block_index = int(
                match.group(1)
            )

            if (
                block_index
                >= min_block
            ):
                p.requires_grad = True
                matched.append(name)

        if (
            "encoder_norm"
            in name
        ):
            p.requires_grad = True
            matched.append(name)

    if not matched:
        raise RuntimeError(
            "Could not identify TerraMind encoder block names. "
            "Stop here and inspect model.backbone.named_parameters()."
        )

    return matched


freeze_entire_backbone(
    model
)

with torch.no_grad():
    tiny_sar = batch["sar"][
        :1
    ].to(
        DEVICE
    )

    tiny_opt = batch["optical"][
        :1
    ].to(
        DEVICE
    )

    tiny_rel = batch["reliability"][
        :1
    ].to(
        DEVICE
    )

    output = model(
        tiny_sar,
        tiny_opt,
        tiny_rel,
        return_aux=True,
    )

print("=" * 112)
print("P7-C08 — ARCHITECTURE SMOKE TEST")
print("=" * 112)

print(
    "Main logits:",
    tuple(
        output[
            "logits"
        ].shape
    ),
)

print(
    "Aux logits :",
    tuple(
        output[
            "aux_logits"
        ].shape
    ),
)

print(
    "Router maps:",
    [
        tuple(x.shape)
        for x in output[
            "router_weights"
        ]
    ],
)

assert (
    output["logits"].shape
    == (
        1,
        NUM_CLASSES,
        256,
        256,
    )
)

assert torch.isfinite(
    output["logits"]
).all()

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params_stage_a = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Total parameters     :",
    f"{total_params/1e6:.2f} M",
)

print(
    "Stage-A trainable    :",
    f"{trainable_params_stage_a/1e6:.2f} M",
)

print(
    "Backbone policy      : "
    "FROZEN for epochs 1-5"
)

del output
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("P7-C08 COMPLETE ✅")

P7-C08 — ARCHITECTURE SMOKE TEST
Main logits: (1, 3, 256, 256)
Aux logits : (1, 3, 64, 64)
Router maps: [(1, 2, 128, 128), (1, 2, 64, 64), (1, 2, 32, 32), (1, 2, 16, 16)]
Total parameters     : 106.68 M
Stage-A trainable    : 19.36 M
Backbone policy      : FROZEN for epochs 1-5
P7-C08 COMPLETE ✅


In [14]:
# ============================================================
# CELL: P7-C09 — HIGH-STANDARD LOSS
# 45% balanced CE + 35% Lovasz + 20% temporary Tversky
# plus small auxiliary CE.
# ============================================================

CE_WEIGHT = 0.45
LOVASZ_WEIGHT = 0.35
TEMP_TVERSKY_WEIGHT = 0.20
AUX_WEIGHT = 0.08

LABEL_SMOOTHING = 0.01

cross_entropy = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS,
    ignore_index=IGNORE_INDEX,
    label_smoothing=LABEL_SMOOTHING,
)


def lovasz_grad(gt_sorted):
    p = len(gt_sorted)

    gts = gt_sorted.sum()

    intersection = (
        gts
        - gt_sorted.float().cumsum(0)
    )

    union = (
        gts
        + (
            1.0 - gt_sorted
        ).float().cumsum(0)
    )

    jaccard = (
        1.0
        - intersection
        / union.clamp_min(1e-7)
    )

    if p > 1:
        jaccard[
            1:p
        ] = (
            jaccard[1:p]
            - jaccard[
                :-1
            ]
        )

    return jaccard


def lovasz_softmax_flat(
    probs,
    labels,
):
    losses = []

    for c in range(
        NUM_CLASSES
    ):
        foreground = (
            labels == c
        ).float()

        if (
            foreground.sum()
            == 0
        ):
            continue

        class_pred = probs[
            :,
            c,
        ]

        errors = (
            foreground
            - class_pred
        ).abs()

        (
            errors_sorted,
            perm,
        ) = torch.sort(
            errors,
            descending=True,
        )

        fg_sorted = foreground[
            perm
        ]

        losses.append(
            torch.dot(
                errors_sorted,
                lovasz_grad(
                    fg_sorted
                ),
            )
        )

    if not losses:
        return probs.sum() * 0.0

    return torch.stack(
        losses
    ).mean()


def lovasz_softmax_loss(
    logits,
    target,
):
    probs = torch.softmax(
        logits,
        dim=1,
    )

    valid = (
        target
        != IGNORE_INDEX
    )

    if (
        valid.sum()
        == 0
    ):
        return logits.sum() * 0.0

    probs_flat = (
        probs
        .permute(
            0,
            2,
            3,
            1,
        )[
            valid
        ]
    )

    labels_flat = target[
        valid
    ]

    return lovasz_softmax_flat(
        probs_flat,
        labels_flat,
    )


def temporary_focal_tversky(
    logits,
    target,
    alpha=0.60,
    beta=0.40,
    gamma=0.75,
):
    valid = (
        target
        != IGNORE_INDEX
    )

    if (
        valid.sum()
        == 0
    ):
        return logits.sum() * 0.0

    prob_temp = torch.softmax(
        logits,
        dim=1,
    )[:, 2]

    gt_temp = (
        target == 2
    ).float()

    p = prob_temp[
        valid
    ]

    g = gt_temp[
        valid
    ]

    tp = (
        p * g
    ).sum()

    fp = (
        p * (
            1.0 - g
        )
    ).sum()

    fn = (
        (
            1.0 - p
        ) * g
    ).sum()

    tversky = (
        tp + 1.0
    ) / (
        tp
        + alpha * fp
        + beta * fn
        + 1.0
    )

    return (
        1.0 - tversky
    ).pow(gamma)


def segmentation_loss(
    output,
    target,
):
    logits = output[
        "logits"
    ]

    aux_logits = output[
        "aux_logits"
    ]

    ce = cross_entropy(
        logits,
        target,
    )

    lovasz = (
        lovasz_softmax_loss(
            logits,
            target,
        )
    )

    temp_tv = (
        temporary_focal_tversky(
            logits,
            target,
        )
    )

    aux_up = F.interpolate(
        aux_logits,
        size=target.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )

    aux_ce = cross_entropy(
        aux_up,
        target,
    )

    total = (
        CE_WEIGHT * ce
        + LOVASZ_WEIGHT * lovasz
        + TEMP_TVERSKY_WEIGHT * temp_tv
        + AUX_WEIGHT * aux_ce
    )

    parts = {
        "ce": float(
            ce.detach()
        ),
        "lovasz": float(
            lovasz.detach()
        ),
        "temp_tversky": float(
            temp_tv.detach()
        ),
        "aux_ce": float(
            aux_ce.detach()
        ),
    }

    return total, parts


with torch.no_grad():
    smoke_output = model(
        tiny_sar,
        tiny_opt,
        tiny_rel,
        return_aux=True,
    )

    smoke_target = batch[
        "target"
    ][
        :1
    ].to(
        DEVICE
    )

    smoke_loss, smoke_parts = (
        segmentation_loss(
            smoke_output,
            smoke_target,
        )
    )

print("Loss smoke test:", float(smoke_loss))
print("Components:", smoke_parts)

assert torch.isfinite(
    smoke_loss
)

del smoke_output
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("P7-C09 COMPLETE ✅")

Loss smoke test: 1.0366480350494385
Components: {'ce': 1.1531769037246704, 'lovasz': 0.6914272308349609, 'temp_tversky': 0.9808051586151123, 'aux_ce': 0.9944734573364258}
P7-C09 COMPLETE ✅


In [15]:
# ============================================================
# CELL: P7-C10 — METRICS + EVALUATION + TTA
# ============================================================

def confusion_from_predictions(
    pred,
    target,
):
    valid = (
        target
        != IGNORE_INDEX
    )

    target_v = target[
        valid
    ].long()

    pred_v = pred[
        valid
    ].long()

    indices = (
        NUM_CLASSES
        * target_v
        + pred_v
    )

    cm = torch.bincount(
        indices,
        minlength=(
            NUM_CLASSES
            * NUM_CLASSES
        ),
    )

    return cm.reshape(
        NUM_CLASSES,
        NUM_CLASSES,
    )


def metrics_from_confusion(cm):
    cm = (
        cm
        .detach()
        .cpu()
        .numpy()
        .astype(np.float64)
    )

    tp = np.diag(cm)

    fp = (
        cm.sum(axis=0)
        - tp
    )

    fn = (
        cm.sum(axis=1)
        - tp
    )

    denom_iou = (
        tp + fp + fn
    )

    iou = np.divide(
        tp,
        denom_iou,
        out=np.full_like(
            tp,
            np.nan,
            dtype=np.float64,
        ),
        where=(
            denom_iou > 0
        ),
    )

    denom_dice = (
        2.0 * tp
        + fp
        + fn
    )

    dice = np.divide(
        2.0 * tp,
        denom_dice,
        out=np.full_like(
            tp,
            np.nan,
            dtype=np.float64,
        ),
        where=(
            denom_dice > 0
        ),
    )

    precision = np.divide(
        tp,
        tp + fp,
        out=np.full_like(
            tp,
            np.nan,
            dtype=np.float64,
        ),
        where=(
            tp + fp > 0
        ),
    )

    recall = np.divide(
        tp,
        tp + fn,
        out=np.full_like(
            tp,
            np.nan,
            dtype=np.float64,
        ),
        where=(
            tp + fn > 0
        ),
    )

    accuracy = (
        tp.sum()
        / max(
            cm.sum(),
            1.0,
        )
    )

    result = {
        "mIoU": float(
            np.nanmean(iou)
        ),
        "macro_dice": float(
            np.nanmean(dice)
        ),
        "accuracy": float(
            accuracy
        ),
        "confusion": cm.tolist(),
    }

    for i, name in enumerate(
        CLASS_NAMES
    ):
        result[
            f"{name}_IoU"
        ] = float(iou[i])

        result[
            f"{name}_Dice"
        ] = float(dice[i])

        result[
            f"{name}_Precision"
        ] = float(
            precision[i]
        )

        result[
            f"{name}_Recall"
        ] = float(
            recall[i]
        )

    return result


@torch.inference_mode()
def evaluate_model(
    model,
    loader,
):
    model.eval()

    cm = torch.zeros(
        (
            NUM_CLASSES,
            NUM_CLASSES,
        ),
        dtype=torch.long,
        device=DEVICE,
    )

    loss_total = 0.0
    batches = 0

    for batch_eval in loader:
        sar = batch_eval[
            "sar"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        optical = batch_eval[
            "optical"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        reliability = batch_eval[
            "reliability"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        target = batch_eval[
            "target"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.cuda.amp.autocast(
            enabled=AMP_ENABLED
        ):
            output = model(
                sar,
                optical,
                reliability,
                return_aux=True,
            )

            loss, _ = (
                segmentation_loss(
                    output,
                    target,
                )
            )

        pred = output[
            "logits"
        ].argmax(
            dim=1
        )

        cm += confusion_from_predictions(
            pred,
            target,
        )

        loss_total += float(
            loss.detach()
        )

        batches += 1

    result = metrics_from_confusion(
        cm
    )

    result[
        "loss"
    ] = (
        loss_total
        / max(
            batches,
            1,
        )
    )

    return result


def _tta_transform(
    tensor,
    mode,
):
    if mode == "identity":
        return tensor

    if mode == "hflip":
        return torch.flip(
            tensor,
            dims=[-1],
        )

    if mode == "vflip":
        return torch.flip(
            tensor,
            dims=[-2],
        )

    if mode == "rot180":
        return torch.rot90(
            tensor,
            k=2,
            dims=(-2, -1),
        )

    raise ValueError(mode)


def _tta_inverse(
    tensor,
    mode,
):
    # All four selected transforms are self-inverse.
    return _tta_transform(
        tensor,
        mode,
    )


@torch.inference_mode()
def evaluate_model_tta(
    model,
    loader,
):
    model.eval()

    modes = [
        "identity",
        "hflip",
        "vflip",
        "rot180",
    ]

    cm = torch.zeros(
        (
            NUM_CLASSES,
            NUM_CLASSES,
        ),
        dtype=torch.long,
        device=DEVICE,
    )

    for batch_eval in loader:
        sar = batch_eval[
            "sar"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        optical = batch_eval[
            "optical"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        reliability = batch_eval[
            "reliability"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        target = batch_eval[
            "target"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        logits_sum = None

        for mode in modes:
            sar_t = _tta_transform(
                sar,
                mode,
            )

            opt_t = _tta_transform(
                optical,
                mode,
            )

            rel_t = _tta_transform(
                reliability,
                mode,
            )

            with torch.cuda.amp.autocast(
                enabled=AMP_ENABLED
            ):
                logits = model(
                    sar_t,
                    opt_t,
                    rel_t,
                    return_aux=False,
                )

            logits = _tta_inverse(
                logits,
                mode,
            )

            if logits_sum is None:
                logits_sum = (
                    logits.float()
                )
            else:
                logits_sum += (
                    logits.float()
                )

        logits_mean = (
            logits_sum
            / len(modes)
        )

        pred = logits_mean.argmax(
            dim=1
        )

        cm += confusion_from_predictions(
            pred,
            target,
        )

    return metrics_from_confusion(
        cm
    )


def compact_metrics(m):
    return {
        "mIoU": round(
            m[
                "mIoU"
            ],
            4,
        ),
        "non_flood": round(
            m[
                "non_flood_IoU"
            ],
            4,
        ),
        "permanent": round(
            m[
                "permanent_water_IoU"
            ],
            4,
        ),
        "temporary": round(
            m[
                "temporary_flood_IoU"
            ],
            4,
        ),
    }


print("P7-C10 COMPLETE ✅")

P7-C10 COMPLETE ✅


In [16]:
# ============================================================
# CELL: P7-C11 — STAGED TRAINING
#
# Stage A, epochs 1-5:
#   TerraMind frozen.
#
# Stage B:
#   only last 2 TerraMind encoder blocks + encoder norm unfreeze.
#
# IMPORTANT:
#   We intentionally NEVER full-unfreeze TerraMind because
#   the previous V3 run collapsed after broad unfreezing.
# ============================================================

MAX_EPOCHS = 30
FROZEN_BACKBONE_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 8

WEIGHT_DECAY = 1e-2
GRAD_CLIP_NORM = 1.0

# Different learning rates are deliberate.
LR_BACKBONE_TOP = 2.0e-6
LR_FUSION_STAGE_A = 2.0e-4
LR_DECODER_STAGE_A = 3.0e-4
LR_HEAD_STAGE_A = 4.0e-4

LR_FUSION_STAGE_B = 8.0e-5
LR_DECODER_STAGE_B = 1.5e-4
LR_HEAD_STAGE_B = 2.0e-4

TOP_K_CHECKPOINTS = 3

HISTORY_PATH = (
    HISTORY_ROOT
    / "final_terramind_rift_history.csv"
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_FINAL_TerraMind_RIFT_best.pt"
)


def module_parameters(
    modules,
):
    params = []

    for module in modules:
        params.extend(
            [
                p
                for p in module.parameters()
                if p.requires_grad
            ]
        )

    return params


def build_optimizer(
    model,
    stage,
):
    if stage == "A":
        freeze_entire_backbone(
            model
        )

        backbone_params = []

        fusion_lr = (
            LR_FUSION_STAGE_A
        )

        decoder_lr = (
            LR_DECODER_STAGE_A
        )

        head_lr = (
            LR_HEAD_STAGE_A
        )

    elif stage == "B":
        matched = (
            unfreeze_last_backbone_blocks(
                model,
                last_n=2,
                depth=12,
            )
        )

        print(
            "Unfrozen TerraMind "
            "parameter tensors:",
            len(matched),
        )

        backbone_params = [
            p
            for p in model.backbone.parameters()
            if p.requires_grad
        ]

        fusion_lr = (
            LR_FUSION_STAGE_B
        )

        decoder_lr = (
            LR_DECODER_STAGE_B
        )

        head_lr = (
            LR_HEAD_STAGE_B
        )

    else:
        raise ValueError(stage)

    fusion_modules = [
        model.sar_projectors,
        model.opt_projectors,
        model.routers,
        model.deep_rift,
        model.deep_context,
    ]

    decoder_modules = [
        model.decoder3,
        model.decoder2,
        model.decoder1,
        model.final_refine,
    ]

    head_modules = [
        model.segmentation_head,
        model.aux_head,
    ]

    param_groups = []

    if backbone_params:
        param_groups.append({
            "params": backbone_params,
            "lr": LR_BACKBONE_TOP,
            "name": "backbone_top",
        })

    param_groups.extend([
        {
            "params": module_parameters(
                fusion_modules
            ),
            "lr": fusion_lr,
            "name": "fusion",
        },
        {
            "params": module_parameters(
                decoder_modules
            ),
            "lr": decoder_lr,
            "name": "decoder",
        },
        {
            "params": module_parameters(
                head_modules
            ),
            "lr": head_lr,
            "name": "heads",
        },
    ])

    optimizer = torch.optim.AdamW(
        param_groups,
        weight_decay=WEIGHT_DECAY,
    )

    return optimizer


def build_scheduler(
    optimizer,
    stage_epochs,
):
    return (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(
                stage_epochs,
                1,
            ),
            eta_min=1e-7,
        )
    )


def save_checkpoint(
    path,
    model,
    epoch,
    metrics,
    stage,
):
    payload = {
        "state_dict": model.state_dict(),
        "epoch": int(epoch),
        "stage": stage,
        "val_metrics": metrics,
        "class_names": CLASS_NAMES,
        "architecture":
            "FloodShift-FINAL-TerraMind-RIFT",
        "target_semantics":
            "valid JRC==1 -> permanent water; "
            "valid flood==1 & JRC!=1 -> temporary flood",
    }

    torch.save(
        payload,
        path,
    )


def is_better(
    current,
    best,
):
    if best is None:
        return True

    if (
        current["mIoU"]
        > best["mIoU"]
        + 1e-7
    ):
        return True

    if (
        abs(
            current["mIoU"]
            - best["mIoU"]
        )
        <= 1e-7
        and current[
            "temporary_flood_IoU"
        ]
        > best[
            "temporary_flood_IoU"
        ]
    ):
        return True

    return False


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        SEED
    )

(
    train_loader,
    val_loader,
    test_loader,
) = make_loaders(
    seed=SEED
)

model = model.to(
    DEVICE
)

stage = "A"

optimizer = build_optimizer(
    model,
    stage=stage,
)

scheduler = build_scheduler(
    optimizer,
    stage_epochs=(
        FROZEN_BACKBONE_EPOCHS
    ),
)

scaler = torch.cuda.amp.GradScaler(
    enabled=AMP_ENABLED
)

best_val = None
best_epoch = None
epochs_without_improvement = 0

history = []
top_records = []

training_start = time.time()

for epoch in range(
    1,
    MAX_EPOCHS + 1,
):
    # --------------------------------------------------------
    # Stage transition. Only one conservative partial unfreeze.
    # --------------------------------------------------------
    if (
        epoch
        == FROZEN_BACKBONE_EPOCHS + 1
    ):
        stage = "B"

        optimizer = build_optimizer(
            model,
            stage=stage,
        )

        scheduler = build_scheduler(
            optimizer,
            stage_epochs=(
                MAX_EPOCHS
                - FROZEN_BACKBONE_EPOCHS
            ),
        )

        print(
            "\n"
            + "=" * 112
        )

        print(
            "STAGE B STARTED — "
            "ONLY FINAL 2 TERRAMIND BLOCKS UNFROZEN"
        )

        print(
            "=" * 112
        )

    model.train()

    # Keep the pretrained TerraMind backbone in deterministic
    # evaluation mode even when its final two blocks are trainable.
    # Gradients still flow through trainable parameters, but stochastic
    # depth/dropout cannot destabilize the small geographic training set.
    model.backbone.eval()

    running_loss = 0.0
    running_steps = 0

    optimizer.zero_grad(
        set_to_none=True
    )

    epoch_start = time.time()

    for step, batch_train in enumerate(
        train_loader,
        start=1,
    ):
        sar = batch_train[
            "sar"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        optical = batch_train[
            "optical"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        reliability = batch_train[
            "reliability"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        target = batch_train[
            "target"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.cuda.amp.autocast(
            enabled=AMP_ENABLED
        ):
            output = model(
                sar,
                optical,
                reliability,
                return_aux=True,
            )

            loss, loss_parts = (
                segmentation_loss(
                    output,
                    target,
                )
            )

            scaled_loss = (
                loss
                / GRAD_ACCUM_STEPS
            )

        scaler.scale(
            scaled_loss
        ).backward()

        do_step = (
            step % GRAD_ACCUM_STEPS
            == 0
            or step
            == len(train_loader)
        )

        if do_step:
            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                [
                    p
                    for p in model.parameters()
                    if p.requires_grad
                ],
                GRAD_CLIP_NORM,
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

        running_loss += float(
            loss.detach()
        )

        running_steps += 1

    train_loss = (
        running_loss
        / max(
            running_steps,
            1,
        )
    )

    val_metrics = evaluate_model(
        model,
        val_loader,
    )

    epoch_seconds = (
        time.time()
        - epoch_start
    )

    lr_snapshot = {
        group.get(
            "name",
            f"group_{i}",
        ): group["lr"]
        for i, group in enumerate(
            optimizer.param_groups
        )
    }

    row = {
        "epoch": epoch,
        "stage": stage,
        "train_loss": train_loss,
        "val_loss": val_metrics[
            "loss"
        ],
        "val_mIoU": val_metrics[
            "mIoU"
        ],
        "val_non_flood_IoU":
            val_metrics[
                "non_flood_IoU"
            ],
        "val_permanent_water_IoU":
            val_metrics[
                "permanent_water_IoU"
            ],
        "val_temporary_flood_IoU":
            val_metrics[
                "temporary_flood_IoU"
            ],
        "epoch_seconds":
            epoch_seconds,
        "learning_rates":
            json.dumps(
                lr_snapshot
            ),
    }

    history.append(row)

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    print(
        f"Epoch {epoch:02d} "
        f"[Stage {stage}] | "
        f"train {train_loss:.4f} | "
        f"val {val_metrics['loss']:.4f} | "
        f"mIoU {val_metrics['mIoU']:.4f} | "
        f"perm {val_metrics['permanent_water_IoU']:.4f} | "
        f"temp {val_metrics['temporary_flood_IoU']:.4f} | "
        f"{epoch_seconds/60:.1f} min"
    )

    # --------------------------------------------------------
    # Maintain top-3 validation checkpoints.
    # --------------------------------------------------------
    top_path = (
        CHECKPOINT_ROOT
        / f"top_epoch_{epoch:02d}.pt"
    )

    save_checkpoint(
        top_path,
        model,
        epoch,
        val_metrics,
        stage,
    )

    top_records.append({
        "path": str(top_path),
        "epoch": epoch,
        "mIoU": val_metrics[
            "mIoU"
        ],
        "temporary":
            val_metrics[
                "temporary_flood_IoU"
            ],
    })

    top_records = sorted(
        top_records,
        key=lambda x: (
            x["mIoU"],
            x["temporary"],
        ),
        reverse=True,
    )

    while (
        len(top_records)
        > TOP_K_CHECKPOINTS
    ):
        removed = top_records.pop()

        removed_path = Path(
            removed["path"]
        )

        if removed_path.exists():
            removed_path.unlink()

    # --------------------------------------------------------
    # Best checkpoint.
    # --------------------------------------------------------
    if is_better(
        val_metrics,
        best_val,
    ):
        best_val = copy.deepcopy(
            val_metrics
        )

        best_epoch = epoch

        epochs_without_improvement = 0

        save_checkpoint(
            BEST_CHECKPOINT_PATH,
            model,
            epoch,
            val_metrics,
            stage,
        )

        print(
            "  ✅ NEW BEST VALIDATION CHECKPOINT"
        )

    else:
        epochs_without_improvement += 1

    scheduler.step()

    # Do not early-stop during frozen warmup.
    if (
        epoch
        > FROZEN_BACKBONE_EPOCHS
        and epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print(
            "Early stopping triggered."
        )
        break

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


training_minutes = (
    time.time()
    - training_start
) / 60.0

assert BEST_CHECKPOINT_PATH.exists()

best_payload = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
)

model.load_state_dict(
    best_payload[
        "state_dict"
    ]
)

best_val = best_payload[
    "val_metrics"
]

best_epoch = int(
    best_payload[
        "epoch"
    ]
)

print("\n" + "=" * 112)
print("TRAINING COMPLETE")
print("=" * 112)
print("Best epoch       :", best_epoch)
print("Best validation  :", compact_metrics(best_val))
print("Training minutes :", f"{training_minutes:.1f}")
print("Top checkpoints  :")

for record in top_records:
    print(
        " ",
        record,
    )

/tmp/ipykernel_167/1739789673.py:291: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(
/tmp/ipykernel_167/1739789673.py:393: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_167/3182540785.py:220: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 01 [Stage A] | train 0.5204 | val 0.4897 | mIoU 0.4985 | perm 0.2150 | temp 0.3209 | 2.1 min
  ✅ NEW BEST VALIDATION CHECKPOINT
Epoch 02 [Stage A] | train 0.4379 | val 0.4712 | mIoU 0.6591 | perm 0.6281 | temp 0.3956 | 1.5 min
  ✅ NEW BEST VALIDATION CHECKPOINT
Epoch 03 [Stage A] | train 0.4176 | val 0.5138 | mIoU 0.4595 | perm 0.0798 | temp 0.3266 | 1.5 min
Epoch 04 [Stage A] | train 0.3949 | val 0.4740 | mIoU 0.6497 | perm 0.5606 | temp 0.4227 | 1.5 min
Epoch 05 [Stage A] | train 0.3860 | val 0.4519 | mIoU 0.6799 | perm 0.6225 | temp 0.4493 | 1.5 min
  ✅ NEW BEST VALIDATION CHECKPOINT
Unfrozen TerraMind parameter tensors: 15

STAGE B STARTED — ONLY FINAL 2 TERRAMIND BLOCKS UNFROZEN
Epoch 06 [Stage B] | train 0.3925 | val 0.4396 | mIoU 0.7286 | perm 0.7232 | temp 0.4889 | 1.6 min
  ✅ NEW BEST VALIDATION CHECKPOINT
Epoch 07 [Stage B] | train 0.3820 | val 0.4440 | mIoU 0.6964 | perm 0.6677 | temp 0.4517 | 1.6 min
Epoch 08 [Stage B] | train 0.3799 | val 0.4736 | mIoU 0.5660 | perm 

In [17]:
# ============================================================
# CHECK TOP-3 CHECKPOINTS FOR NaN / Inf PARAMETERS
# ============================================================

import torch
import gc

paths = [
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_17.pt",
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_15.pt",
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_12.pt",
]

def extract_state_dict(ckpt):
    possible_keys = [
        "model_state_dict",
        "state_dict",
        "model_state",
        "model",
    ]

    for key in possible_keys:
        if key in ckpt and isinstance(ckpt[key], dict):
            return ckpt[key], key

    # checkpoint itself may be state_dict
    if isinstance(ckpt, dict):
        vals = list(ckpt.values())
        if vals and all(torch.is_tensor(v) for v in vals):
            return ckpt, "<root>"

    raise RuntimeError(
        f"Could not find model state_dict. Keys: {list(ckpt.keys())}"
    )


print("=" * 110)
print("TOP CHECKPOINT FINITE-WEIGHT AUDIT")
print("=" * 110)

all_clean = True

for path in paths:
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    state, state_key = extract_state_dict(ckpt)

    bad = []
    floating_tensors = 0
    total_parameters = 0

    for name, tensor in state.items():

        if not torch.is_tensor(tensor):
            continue

        total_parameters += tensor.numel()

        if torch.is_floating_point(tensor):
            floating_tensors += 1

            if not torch.isfinite(tensor).all():
                bad.append(name)

    print("\n", path.split("/")[-1])
    print("  State key       :", state_key)
    print("  Floating tensors:", floating_tensors)
    print("  Parameters      :", f"{total_parameters:,}")

    if bad:
        all_clean = False
        print("  ❌ NON-FINITE tensors:", len(bad))

        for name in bad[:20]:
            print("    ", name)

    else:
        print("  ✅ ALL PARAMETERS FINITE")

    del ckpt, state
    gc.collect()


print("\n" + "=" * 110)

if all_clean:
    print("✅ TOP-3 CHECKPOINTS ARE NUMERICALLY CLEAN")
else:
    print("❌ STOP — DO NOT AVERAGE CHECKPOINTS")

print("=" * 110)

TOP CHECKPOINT FINITE-WEIGHT AUDIT

 top_epoch_17.pt
  State key       : state_dict
  Floating tensors: 408
  Parameters      : 107,091,462
  ❌ NON-FINITE tensors: 1
     backbone.missing_mod_token

 top_epoch_15.pt
  State key       : state_dict
  Floating tensors: 408
  Parameters      : 107,091,462
  ❌ NON-FINITE tensors: 1
     backbone.missing_mod_token

 top_epoch_12.pt
  State key       : state_dict
  Floating tensors: 408
  Parameters      : 107,091,462
  ❌ NON-FINITE tensors: 1
     backbone.missing_mod_token

❌ STOP — DO NOT AVERAGE CHECKPOINTS


In [19]:
# ============================================================
# DEFINITIVE AUDIT:
# DOES TerraMind missing_mod_token AFFECT OUR S1+S2 FORWARD?
# ============================================================

import torch
import numpy as np

model.eval()

def get_main_logits(output):
    """Robustly extract the main segmentation logits."""
    
    if torch.is_tensor(output):
        return output

    if isinstance(output, dict):
        for key in [
            "logits",
            "main_logits",
            "main",
            "out",
            "segmentation",
        ]:
            if key in output and torch.is_tensor(output[key]):
                return output[key]

        # fallback: first 4D tensor with 3 classes
        for value in output.values():
            if (
                torch.is_tensor(value)
                and value.ndim == 4
                and value.shape[1] == 3
            ):
                return value

    if isinstance(output, (tuple, list)):
        for value in output:
            if (
                torch.is_tensor(value)
                and value.ndim == 4
                and value.shape[1] == 3
            ):
                return value

    raise RuntimeError(
        f"Could not identify main logits from output type {type(output)}"
    )


def forward_validation_logits():
    all_logits = []

    with torch.no_grad():
        for batch in val_loader:

            sar = batch["sar"].to(DEVICE, non_blocking=True)
            optical = batch["optical"].to(DEVICE, non_blocking=True)
            reliability = batch["reliability"].to(
                DEVICE,
                non_blocking=True,
            )

            output = model(
                sar,
                optical,
                reliability,
            )

            logits = get_main_logits(output)

            if not torch.isfinite(logits).all():
                raise RuntimeError(
                    "❌ Non-finite segmentation logits detected"
                )

            all_logits.append(
                logits.detach().float().cpu()
            )

    return torch.cat(all_logits, dim=0)


print("=" * 110)
print("MISSING TOKEN — FULL VALIDATION FORWARD AUDIT")
print("=" * 110)

token = model.backbone.missing_mod_token

print("Token shape :", tuple(token.shape))
print("Token finite:", bool(torch.isfinite(token).all()))
print("NaNs        :", int(torch.isnan(token).sum()))

# ------------------------------------------------------------------
# A. Predictions with checkpoint exactly as saved
# ------------------------------------------------------------------

print("\n[A] Running validation with ORIGINAL NaN missing_mod_token...")

logits_before = forward_validation_logits()

print("✅ Original validation forward complete")
print("Shape:", tuple(logits_before.shape))


# ------------------------------------------------------------------
# B. Zero ONLY missing_mod_token
# ------------------------------------------------------------------

print("\n[B] Zeroing ONLY backbone.missing_mod_token...")

with torch.no_grad():
    model.backbone.missing_mod_token.zero_()

assert torch.isfinite(
    model.backbone.missing_mod_token
).all()

print("✅ missing_mod_token is now finite")


# ------------------------------------------------------------------
# C. Predictions again
# ------------------------------------------------------------------

print("\n[C] Running validation after zeroing token...")

logits_after = forward_validation_logits()

print("✅ Sanitized validation forward complete")


# ------------------------------------------------------------------
# D. Exact comparison
# ------------------------------------------------------------------

abs_diff = torch.abs(
    logits_before - logits_after
)

max_diff = abs_diff.max().item()
mean_diff = abs_diff.mean().item()

same_predictions = torch.equal(
    logits_before.argmax(dim=1),
    logits_after.argmax(dim=1),
)

print("\n" + "=" * 110)
print("RESULT")
print("=" * 110)

print(f"Maximum logit difference : {max_diff:.12e}")
print(f"Mean logit difference    : {mean_diff:.12e}")
print(f"Class predictions equal  : {same_predictions}")

if max_diff == 0.0 and same_predictions:
    print(
        "\n✅ PROVEN: missing_mod_token is NOT used "
        "by our complete S1+S2 FloodShift validation path."
    )
else:
    print(
        "\n⚠️ Token affects the model path. "
        "STOP before checkpoint sanitation."
    )

print("=" * 110)

MISSING TOKEN — FULL VALIDATION FORWARD AUDIT
Token shape : (768,)
Token finite: False
NaNs        : 85

[A] Running validation with ORIGINAL NaN missing_mod_token...
✅ Original validation forward complete
Shape: (272, 3, 256, 256)

[B] Zeroing ONLY backbone.missing_mod_token...
✅ missing_mod_token is now finite

[C] Running validation after zeroing token...
✅ Sanitized validation forward complete

RESULT
Maximum logit difference : 0.000000000000e+00
Mean logit difference    : 0.000000000000e+00
Class predictions equal  : True

✅ PROVEN: missing_mod_token is NOT used by our complete S1+S2 FloodShift validation path.


In [20]:
# ============================================================
# SANITIZE UNUSED TERRAMIND missing_mod_token
# Preserve original checkpoints, then zero ONLY this tensor.
# ============================================================

import os
import shutil
import torch

paths = [
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_17.pt",
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_15.pt",
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_12.pt",
]

TOKEN_KEY = "backbone.missing_mod_token"

print("=" * 110)
print("SANITIZING UNUSED TERRAMIND FALLBACK TOKEN")
print("=" * 110)

for path in paths:

    # --------------------------------------------------------
    # 1. Preserve untouched original checkpoint
    # --------------------------------------------------------
    backup_path = path.replace(
        ".pt",
        "_ORIGINAL_UNSANITIZED.pt",
    )

    if not os.path.exists(backup_path):
        shutil.copy2(path, backup_path)
        print("\nBackup created:")
        print(" ", backup_path)
    else:
        print("\nBackup already exists:")
        print(" ", backup_path)

    # --------------------------------------------------------
    # 2. Load working checkpoint
    # --------------------------------------------------------
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    state = ckpt["state_dict"]

    assert TOKEN_KEY in state, (
        f"{TOKEN_KEY} not found in {path}"
    )

    token = state[TOKEN_KEY]

    print("\nCheckpoint:", os.path.basename(path))
    print("  token shape       :", tuple(token.shape))
    print("  finite before     :", bool(torch.isfinite(token).all()))
    print("  NaNs before       :", int(torch.isnan(token).sum()))
    print("  Infs before       :", int(torch.isinf(token).sum()))

    # --------------------------------------------------------
    # 3. Replace ONLY unused fallback token with zeros
    # --------------------------------------------------------
    state[TOKEN_KEY] = torch.zeros_like(token)

    assert torch.isfinite(
        state[TOKEN_KEY]
    ).all()

    # Record exactly what was done.
    ckpt["floodshift_checkpoint_sanitation"] = {
        "parameter": TOKEN_KEY,
        "action": "replaced_with_zeros",
        "reason": (
            "TerraTorch missing-modality fallback parameter "
            "contained non-finite random initialization. "
            "Full validation forward audit proved this parameter "
            "is unused when both S1GRD and S2L1C are supplied: "
            "max logit difference after zeroing = 0."
        ),
    }

    # --------------------------------------------------------
    # 4. Save sanitized working checkpoint
    # --------------------------------------------------------
    torch.save(
        ckpt,
        path,
    )

    print("  ✅ sanitized checkpoint saved")

print("\n" + "=" * 110)
print("✅ SANITATION COMPLETE")
print("Original checkpoints preserved separately.")
print("=" * 110)

SANITIZING UNUSED TERRAMIND FALLBACK TOKEN

Backup created:
  /kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_17_ORIGINAL_UNSANITIZED.pt

Checkpoint: top_epoch_17.pt
  token shape       : (768,)
  finite before     : False
  NaNs before       : 85
  Infs before       : 0
  ✅ sanitized checkpoint saved

Backup created:
  /kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_15_ORIGINAL_UNSANITIZED.pt

Checkpoint: top_epoch_15.pt
  token shape       : (768,)
  finite before     : False
  NaNs before       : 85
  Infs before       : 0
  ✅ sanitized checkpoint saved

Backup created:
  /kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_12_ORIGINAL_UNSANITIZED.pt

Checkpoint: top_epoch_12.pt
  token shape       : (768,)
  finite before     : False
  NaNs before       : 85
  Infs before       : 0
  ✅ sanitized checkpoint saved

✅ SANITATION COMPLETE
Original checkpoints preserved separately.


In [21]:
import torch
import os

paths = [
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_17.pt",
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_15.pt",
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/top_epoch_12.pt",
]

all_clean = True

print("=" * 110)
print("POST-SANITATION CHECKPOINT AUDIT")
print("=" * 110)

for path in paths:

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    state = ckpt["state_dict"]

    bad = []

    for name, tensor in state.items():

        if (
            torch.is_tensor(tensor)
            and torch.is_floating_point(tensor)
            and not torch.isfinite(tensor).all()
        ):
            bad.append(name)

    print("\n", os.path.basename(path))

    if bad:
        all_clean = False
        print("  ❌ Non-finite tensors:", bad)
    else:
        print("  ✅ ALL PARAMETERS FINITE")

print("\n" + "=" * 110)

if all_clean:
    print("✅ TOP-3 CHECKPOINTS ARE FULLY FINITE")
else:
    print("❌ STOP")

print("=" * 110)

POST-SANITATION CHECKPOINT AUDIT

 top_epoch_17.pt
  ✅ ALL PARAMETERS FINITE

 top_epoch_15.pt
  ✅ ALL PARAMETERS FINITE

 top_epoch_12.pt
  ✅ ALL PARAMETERS FINITE

✅ TOP-3 CHECKPOINTS ARE FULLY FINITE


In [22]:
# ============================================================
# CELL: P7-C12 — VALIDATION-ONLY TOP-3 WEIGHT AVERAGING
#
# We keep the averaged model ONLY if validation mIoU improves.
# Test remains untouched.
# ============================================================

top_paths = [
    Path(
        x["path"]
    )
    for x in top_records
    if Path(
        x["path"]
    ).exists()
]

assert len(top_paths) >= 1

single_payload = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
)

single_state = single_payload[
    "state_dict"
]

single_val = single_payload[
    "val_metrics"
]

chosen_kind = "best_single"
chosen_checkpoint_path = (
    BEST_CHECKPOINT_PATH
)

AVERAGED_CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_FINAL_TerraMind_RIFT_top3_avg.pt"
)

if len(top_paths) >= 2:
    averaged_state = None

    for i, path in enumerate(
        top_paths
    ):
        payload = torch.load(
            path,
            map_location="cpu",
        )

        state = payload[
            "state_dict"
        ]

        if averaged_state is None:
            averaged_state = {}

            for k, v in state.items():
                if (
                    torch.is_floating_point(v)
                    or torch.is_complex(v)
                ):
                    averaged_state[k] = (
                        v.float()
                        / len(top_paths)
                    )
                else:
                    averaged_state[k] = (
                        v.clone()
                    )

        else:
            for k, v in state.items():
                if (
                    torch.is_floating_point(v)
                    or torch.is_complex(v)
                ):
                    averaged_state[k] += (
                        v.float()
                        / len(top_paths)
                    )

        del payload
        del state
        gc.collect()

    # Restore dtypes to the reference state dict.
    for k in averaged_state:
        averaged_state[k] = (
            averaged_state[k]
            .to(
                dtype=single_state[
                    k
                ].dtype
            )
        )

    model.load_state_dict(
        averaged_state
    )

    model = model.to(
        DEVICE
    )

    averaged_val = evaluate_model(
        model,
        val_loader,
    )

    print(
        "Best single:",
        compact_metrics(
            single_val
        ),
    )

    print(
        "Top-k average:",
        compact_metrics(
            averaged_val
        ),
    )

    # Require a real gain, not numerical noise.
    if (
        averaged_val["mIoU"]
        >= single_val["mIoU"]
        + 0.001
    ):
        torch.save(
            {
                "state_dict":
                    model.state_dict(),
                "val_metrics":
                    averaged_val,
                "source_checkpoints":
                    [
                        str(p)
                        for p in top_paths
                    ],
                "architecture":
                    "FloodShift-FINAL-TerraMind-RIFT",
            },
            AVERAGED_CHECKPOINT_PATH,
        )

        chosen_kind = (
            "top_checkpoint_weight_average"
        )

        chosen_checkpoint_path = (
            AVERAGED_CHECKPOINT_PATH
        )

        chosen_val = (
            averaged_val
        )

        print(
            "✅ Validation selected "
            "the weight-averaged model."
        )

    else:
        model.load_state_dict(
            single_state
        )

        model = model.to(
            DEVICE
        )

        chosen_val = (
            single_val
        )

        print(
            "✅ Validation retained "
            "the best single checkpoint."
        )

else:
    model.load_state_dict(
        single_state
    )

    model = model.to(
        DEVICE
    )

    chosen_val = (
        single_val
    )

    print(
        "Only one top checkpoint exists; "
        "using best single."
    )

del single_state
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "Chosen model:",
    chosen_kind,
)

print(
    "Chosen validation:",
    compact_metrics(
        chosen_val
    ),
)

/tmp/ipykernel_167/3182540785.py:220: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Best single: {'mIoU': 0.7743, 'non_flood': 0.9695, 'permanent': 0.8246, 'temporary': 0.5288}
Top-k average: {'mIoU': 0.761, 'non_flood': 0.9691, 'permanent': 0.7995, 'temporary': 0.5144}
✅ Validation retained the best single checkpoint.
Chosen model: best_single
Chosen validation: {'mIoU': 0.7743, 'non_flood': 0.9695, 'permanent': 0.8246, 'temporary': 0.5288}


In [23]:
# ============================================================
# CELL: P7-C13 — VALIDATION-ONLY TTA SELECTION + PROMOTION GATE
#
# TTA is accepted only if validation mIoU improves by >= 0.001.
#
# Promotion thresholds were declared BEFORE final test:
#   mIoU >= 0.560
#   temporary-flood IoU >= 0.350
#   permanent-water IoU >= 0.300
# ============================================================

TTA_MIN_GAIN = 0.001

PROMOTION_MIN_MIOU = 0.560
PROMOTION_MIN_TEMP_IOU = 0.350
PROMOTION_MIN_PERM_IOU = 0.300

print("Evaluating validation with 4-way TTA...")

tta_val = evaluate_model_tta(
    model,
    val_loader,
)

print(
    "No-TTA validation:",
    compact_metrics(
        chosen_val
    ),
)

print(
    "TTA validation   :",
    compact_metrics(
        tta_val
    ),
)

use_tta = (
    tta_val["mIoU"]
    >= chosen_val["mIoU"]
    + TTA_MIN_GAIN
)

if use_tta:
    final_val = tta_val
    inference_mode = "4way_tta"
else:
    final_val = chosen_val
    inference_mode = "single_view"

promotion_pass = (
    final_val["mIoU"]
    >= PROMOTION_MIN_MIOU
    and final_val[
        "temporary_flood_IoU"
    ]
    >= PROMOTION_MIN_TEMP_IOU
    and final_val[
        "permanent_water_IoU"
    ]
    >= PROMOTION_MIN_PERM_IOU
)

print("\n" + "=" * 112)
print("P7-C13 — VALIDATION PROMOTION GATE")
print("=" * 112)

print(
    "Selected inference:",
    inference_mode,
)

print(
    "Final validation:",
    compact_metrics(
        final_val
    ),
)

print(
    "Gate thresholds : "
    f"mIoU>={PROMOTION_MIN_MIOU:.3f}, "
    f"temp>={PROMOTION_MIN_TEMP_IOU:.3f}, "
    f"perm>={PROMOTION_MIN_PERM_IOU:.3f}"
)

print(
    "PROMOTION:",
    "PASS ✅"
    if promotion_pass
    else "FAIL — TEST REMAINS LOCKED 🔒",
)

Evaluating validation with 4-way TTA...


/tmp/ipykernel_167/3182540785.py:381: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


No-TTA validation: {'mIoU': 0.7743, 'non_flood': 0.9695, 'permanent': 0.8246, 'temporary': 0.5288}
TTA validation   : {'mIoU': 0.7791, 'non_flood': 0.9713, 'permanent': 0.8244, 'temporary': 0.5416}

P7-C13 — VALIDATION PROMOTION GATE
Selected inference: 4way_tta
Final validation: {'mIoU': 0.7791, 'non_flood': 0.9713, 'permanent': 0.8244, 'temporary': 0.5416}
Gate thresholds : mIoU>=0.560, temp>=0.350, perm>=0.300
PROMOTION: PASS ✅


In [24]:
# ============================================================
# CELL: P7-C14 — FINAL TEST
#
# ONE-GO RULE:
# The test runs automatically ONLY if the predeclared
# validation promotion gate passed.
#
# No test metric is used to alter model/training choices.
# ============================================================

AUTO_TEST_IF_PROMOTED = True

test_metrics = None

if (
    AUTO_TEST_IF_PROMOTED
    and promotion_pass
):
    print(
        "Promotion gate passed. "
        "Running the held-out test ONCE..."
    )

    if use_tta:
        test_metrics = (
            evaluate_model_tta(
                model,
                test_loader,
            )
        )
    else:
        test_metrics = (
            evaluate_model(
                model,
                test_loader,
            )
        )

    print("\n" + "=" * 112)
    print("FINAL HELD-OUT TEST RESULT")
    print("=" * 112)

    print(
        compact_metrics(
            test_metrics
        )
    )

    with open(
        METRIC_ROOT
        / "final_test_metrics.json",
        "w",
    ) as f:
        json.dump(
            test_metrics,
            f,
            indent=2,
        )

else:
    print("=" * 112)
    print("FINAL TEST NOT RUN 🔒")
    print("=" * 112)

    print(
        "Reason:",
        (
            "promotion gate failed"
            if not promotion_pass
            else
            "AUTO_TEST_IF_PROMOTED=False"
        ),
    )

    print(
        "Validation:",
        compact_metrics(
            final_val
        ),
    )

Promotion gate passed. Running the held-out test ONCE...


/tmp/ipykernel_167/3182540785.py:381: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



FINAL HELD-OUT TEST RESULT
{'mIoU': 0.7162, 'non_flood': 0.9628, 'permanent': 0.6183, 'temporary': 0.5675}


In [25]:
# ============================================================
# CELL: P7-C15 — VALIDATION TEMPERATURE CALIBRATION
#
# Calibration NEVER changes the segmentation mask used for
# model selection. It is post-hoc probability calibration only.
# ============================================================

MAX_CALIBRATION_PIXELS = 200_000


@torch.inference_mode()
def collect_calibration_samples(
    model,
    loader,
    max_pixels=MAX_CALIBRATION_PIXELS,
):
    model.eval()

    logits_cpu = []
    labels_cpu = []

    rng = np.random.default_rng(
        SEED
    )

    for batch_cal in loader:
        sar = batch_cal[
            "sar"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        optical = batch_cal[
            "optical"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        reliability = batch_cal[
            "reliability"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        target = batch_cal[
            "target"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        # Calibration uses selected inference without TTA
        # to keep collection tractable. This calibrates the
        # underlying model probability scale.
        logits = model(
            sar,
            optical,
            reliability,
            return_aux=False,
        ).float()

        valid = (
            target
            != IGNORE_INDEX
        )

        logits_v = (
            logits
            .permute(
                0,
                2,
                3,
                1,
            )[
                valid
            ]
        )

        target_v = target[
            valid
        ]

        if (
            logits_v.shape[0]
            > 4096
        ):
            indices = rng.choice(
                logits_v.shape[0],
                size=4096,
                replace=False,
            )

            indices = torch.tensor(
                indices,
                device=logits_v.device,
                dtype=torch.long,
            )

            logits_v = logits_v[
                indices
            ]

            target_v = target_v[
                indices
            ]

        logits_cpu.append(
            logits_v.cpu()
        )

        labels_cpu.append(
            target_v.cpu()
        )

        current = sum(
            x.shape[0]
            for x in logits_cpu
        )

        if current >= max_pixels:
            break

    logits_all = torch.cat(
        logits_cpu,
        dim=0,
    )[
        :max_pixels
    ]

    labels_all = torch.cat(
        labels_cpu,
        dim=0,
    )[
        :max_pixels
    ]

    return (
        logits_all,
        labels_all,
    )


cal_logits, cal_labels = (
    collect_calibration_samples(
        model,
        val_loader,
    )
)

log_temperature = torch.zeros(
    1,
    device=DEVICE,
    requires_grad=True,
)

cal_logits_gpu = cal_logits.to(
    DEVICE
)

cal_labels_gpu = cal_labels.to(
    DEVICE
)

optimizer_temp = torch.optim.LBFGS(
    [log_temperature],
    lr=0.05,
    max_iter=50,
)


def temp_closure():
    optimizer_temp.zero_grad()

    temperature = (
        log_temperature.exp()
        .clamp(
            0.05,
            20.0,
        )
    )

    loss = F.cross_entropy(
        cal_logits_gpu
        / temperature,
        cal_labels_gpu,
    )

    loss.backward()

    return loss


optimizer_temp.step(
    temp_closure
)

temperature = float(
    log_temperature
    .exp()
    .detach()
    .clamp(
        0.05,
        20.0,
    )
    .cpu()
)

calibration_record = {
    "temperature": temperature,
    "calibration_split": "validation_only",
    "n_pixels": int(
        cal_labels.shape[0]
    ),
    "note":
        "post-hoc probability calibration; "
        "not used for checkpoint selection",
}

with open(
    CALIBRATION_ROOT
    / "temperature_scaling.json",
    "w",
) as f:
    json.dump(
        calibration_record,
        f,
        indent=2,
    )

print(
    "Temperature:",
    temperature,
)

print(
    "Calibration pixels:",
    len(cal_labels),
)

print("P7-C15 COMPLETE ✅")

del (
    cal_logits_gpu,
    cal_labels_gpu,
    cal_logits,
    cal_labels,
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Temperature: 0.9162433743476868
Calibration pixels: 200000
P7-C15 COMPLETE ✅


In [26]:
# ============================================================
# CELL: P7-C16 — SAVE FINAL PROTOCOL / RESULT REGISTRY
# ============================================================

history_df = pd.read_csv(
    HISTORY_PATH
)

protocol = {
    "model":
        "FloodShift-FINAL-TerraMind-RIFT",
    "backbone":
        "terramind_v1_base",
    "backbone_pretrained":
        True,
    "backbone_modalities": [
        "S2L1C",
        "S1GRD",
    ],
    "optical_bands":
        OPTICAL_BANDS,
    "sar_channels": [
        "VV",
        "VH",
    ],
    "terramind_layers":
        TM_LAYER_INDICES,
    "fusion":
        "deep zero-init reliability-conditioned "
        "cross-attention + 4 spatial P(clear) routers",
    "decoder":
        "multiscale routed U-Net / ConvNeXt refine",
    "loss": {
        "weighted_ce":
            CE_WEIGHT,
        "lovasz_softmax":
            LOVASZ_WEIGHT,
        "temporary_focal_tversky":
            TEMP_TVERSKY_WEIGHT,
        "auxiliary_ce":
            AUX_WEIGHT,
    },
    "training": {
        "batch_size":
            BATCH_SIZE,
        "grad_accum_steps":
            GRAD_ACCUM_STEPS,
        "effective_batch":
            BATCH_SIZE
            * GRAD_ACCUM_STEPS,
        "max_epochs":
            MAX_EPOCHS,
        "frozen_backbone_epochs":
            FROZEN_BACKBONE_EPOCHS,
        "partial_unfreeze":
            "last 2 TerraMind encoder blocks only",
        "full_unfreeze":
            False,
        "optimizer":
            "AdamW",
        "weight_decay":
            WEIGHT_DECAY,
        "lr_backbone_top":
            LR_BACKBONE_TOP,
        "lr_fusion_stage_b":
            LR_FUSION_STAGE_B,
        "lr_decoder_stage_b":
            LR_DECODER_STAGE_B,
        "lr_head_stage_b":
            LR_HEAD_STAGE_B,
        "scheduler":
            "CosineAnnealingLR per stage",
        "augmentation":
            "aligned D4 only",
    },
    "data": {
        "dataset":
            "Sen1Floods11 v1.1 HandLabeled",
        "retained_scenes":
            440,
        "train_scenes":
            314,
        "validation_scenes":
            68,
        "test_scenes":
            58,
        "tile_size":
            256,
        "tiles_per_scene":
            4,
        "target_semantics":
            "valid JRC==1 -> class1 permanent_water; "
            "valid flood==1 & JRC!=1 -> class2 temporary_flood; "
            "flood==-1 -> ignore",
        "test_used_for_selection":
            False,
    },
    "selection": {
        "best_epoch":
            best_epoch,
        "chosen_model_kind":
            chosen_kind,
        "inference_mode":
            inference_mode,
        "validation":
            final_val,
        "promotion_pass":
            bool(
                promotion_pass
            ),
        "promotion_thresholds": {
            "mIoU":
                PROMOTION_MIN_MIOU,
            "temporary_flood_IoU":
                PROMOTION_MIN_TEMP_IOU,
            "permanent_water_IoU":
                PROMOTION_MIN_PERM_IOU,
        },
    },
    "test_metrics":
        test_metrics,
    "temperature_scaling":
        calibration_record,
    "training_minutes":
        training_minutes,
    "total_parameters":
        int(
            total_params
        ),
    "history_rows":
        int(
            len(history_df)
        ),
}

PROTOCOL_PATH = (
    OUTPUT_ROOT
    / "FloodShift_FINAL_TerraMind_RIFT_protocol.json"
)

with open(
    PROTOCOL_PATH,
    "w",
) as f:
    json.dump(
        protocol,
        f,
        indent=2,
    )

print("=" * 112)
print("P7-C16 — FINAL EXPORT")
print("=" * 112)
print("Best checkpoint :", BEST_CHECKPOINT_PATH)
print("Chosen checkpoint:", chosen_checkpoint_path)
print("History         :", HISTORY_PATH)
print("Protocol        :", PROTOCOL_PATH)
print("Validation      :", compact_metrics(final_val))
print("Promotion       :", promotion_pass)

if test_metrics is not None:
    print(
        "Test            :",
        compact_metrics(
            test_metrics
        ),
    )
else:
    print(
        "Test            : LOCKED / NOT RUN"
    )

print("\nNOTE:")
print(
    "No architecture can guarantee a literature-best result in one run. "
    "This notebook is the predeclared, performance-first final run."
)

P7-C16 — FINAL EXPORT
Best checkpoint : /kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/FloodShift_FINAL_TerraMind_RIFT_best.pt
Chosen checkpoint: /kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/FloodShift_FINAL_TerraMind_RIFT_best.pt
History         : /kaggle/working/FloodShift_FINAL_TerraMind_RIFT/history/final_terramind_rift_history.csv
Protocol        : /kaggle/working/FloodShift_FINAL_TerraMind_RIFT/FloodShift_FINAL_TerraMind_RIFT_protocol.json
Validation      : {'mIoU': 0.7791, 'non_flood': 0.9713, 'permanent': 0.8244, 'temporary': 0.5416}
Promotion       : True
Test            : {'mIoU': 0.7162, 'non_flood': 0.9628, 'permanent': 0.6183, 'temporary': 0.5675}

NOTE:
No architecture can guarantee a literature-best result in one run. This notebook is the predeclared, performance-first final run.


In [27]:
import torch
import os
import hashlib
import json

FINAL_CKPT = (
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/"
    "checkpoints/FloodShift_FINAL_TerraMind_RIFT_best.pt"
)

PROTOCOL = (
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/"
    "FloodShift_FINAL_TerraMind_RIFT_protocol.json"
)

print("=" * 100)
print("FINAL EXPORT INTEGRITY AUDIT")
print("=" * 100)

ckpt = torch.load(
    FINAL_CKPT,
    map_location="cpu",
    weights_only=False,
)

state = ckpt["state_dict"]

bad = []

for name, tensor in state.items():
    if (
        torch.is_tensor(tensor)
        and torch.is_floating_point(tensor)
        and not torch.isfinite(tensor).all()
    ):
        bad.append(name)

print("Checkpoint:", FINAL_CKPT)
print("Non-finite tensors:", bad)

# SHA256
sha = hashlib.sha256()

with open(FINAL_CKPT, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha.update(chunk)

print("SHA256:", sha.hexdigest())

with open(PROTOCOL, "r") as f:
    protocol = json.load(f)

print("\nProtocol loaded ✅")
print("Protocol keys:", list(protocol.keys()))

print("\n" + "=" * 100)

if not bad:
    print("✅ FINAL EXPORTED CHECKPOINT IS FULLY FINITE")
else:
    print("⚠️ FINAL CHECKPOINT STILL NEEDS THE UNUSED TOKEN SANITIZED")

print("=" * 100)

FINAL EXPORT INTEGRITY AUDIT
Checkpoint: /kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/FloodShift_FINAL_TerraMind_RIFT_best.pt
Non-finite tensors: ['backbone.missing_mod_token']
SHA256: cf822c392e8b32ea74a6ad1d2b98ec479aad01de793d355d947d5056e66be97f

Protocol loaded ✅
Protocol keys: ['model', 'backbone', 'backbone_pretrained', 'backbone_modalities', 'optical_bands', 'sar_channels', 'terramind_layers', 'fusion', 'decoder', 'loss', 'training', 'data', 'selection', 'test_metrics', 'temperature_scaling', 'training_minutes', 'total_parameters', 'history_rows']

⚠️ FINAL CHECKPOINT STILL NEEDS THE UNUSED TOKEN SANITIZED


In [28]:
# ============================================================
# FINAL EXPORTED CHECKPOINT SANITATION
# ============================================================

import os
import shutil
import torch
import hashlib
import json

FINAL_CKPT = (
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/"
    "checkpoints/FloodShift_FINAL_TerraMind_RIFT_best.pt"
)

PROTOCOL = (
    "/kaggle/working/FloodShift_FINAL_TerraMind_RIFT/"
    "FloodShift_FINAL_TerraMind_RIFT_protocol.json"
)

TOKEN_KEY = "backbone.missing_mod_token"

print("=" * 105)
print("FINAL EXPORT SANITATION")
print("=" * 105)

# ------------------------------------------------------------
# 1. Preserve original exported checkpoint
# ------------------------------------------------------------

backup = FINAL_CKPT.replace(
    ".pt",
    "_ORIGINAL_UNSANITIZED.pt",
)

if not os.path.exists(backup):
    shutil.copy2(FINAL_CKPT, backup)
    print("Backup created:")
    print(" ", backup)
else:
    print("Backup already exists:")
    print(" ", backup)

# ------------------------------------------------------------
# 2. Load final checkpoint
# ------------------------------------------------------------

ckpt = torch.load(
    FINAL_CKPT,
    map_location="cpu",
    weights_only=False,
)

state = ckpt["state_dict"]

assert TOKEN_KEY in state

token = state[TOKEN_KEY]

print("\nBefore sanitation")
print("Shape       :", tuple(token.shape))
print("Finite      :", bool(torch.isfinite(token).all()))
print("NaNs        :", int(torch.isnan(token).sum()))
print("Infs        :", int(torch.isinf(token).sum()))

# ------------------------------------------------------------
# 3. Sanitize ONLY unused fallback token
# ------------------------------------------------------------

state[TOKEN_KEY] = torch.zeros_like(token)

assert torch.isfinite(state[TOKEN_KEY]).all()

ckpt["floodshift_checkpoint_sanitation"] = {
    "parameter": TOKEN_KEY,
    "action": "replaced_with_zeros",
    "reason": (
        "TerraMind missing-modality fallback token contained "
        "non-finite initialization. Full validation forward audit "
        "with both S1GRD and S2L1C proved zero effect on logits: "
        "maximum logit difference = 0."
    ),
}

torch.save(
    ckpt,
    FINAL_CKPT,
)

print("\n✅ Final checkpoint sanitized")

# ------------------------------------------------------------
# 4. Re-audit every floating tensor
# ------------------------------------------------------------

ckpt2 = torch.load(
    FINAL_CKPT,
    map_location="cpu",
    weights_only=False,
)

bad = []

for name, tensor in ckpt2["state_dict"].items():

    if (
        torch.is_tensor(tensor)
        and torch.is_floating_point(tensor)
        and not torch.isfinite(tensor).all()
    ):
        bad.append(name)

print("\nNon-finite tensors:", bad)

# ------------------------------------------------------------
# 5. NEW SHA256
# ------------------------------------------------------------

sha = hashlib.sha256()

with open(FINAL_CKPT, "rb") as f:
    for chunk in iter(
        lambda: f.read(1024 * 1024),
        b"",
    ):
        sha.update(chunk)

new_sha = sha.hexdigest()

print("NEW SHA256:", new_sha)

# ------------------------------------------------------------
# 6. Add sanitation + hash record to protocol JSON
# ------------------------------------------------------------

with open(PROTOCOL, "r") as f:
    protocol = json.load(f)

protocol["checkpoint_integrity"] = {
    "final_checkpoint": FINAL_CKPT,
    "sha256": new_sha,
    "all_floating_tensors_finite": len(bad) == 0,
    "sanitized_parameter": TOKEN_KEY,
    "sanitation_behavioral_effect": (
        "none; full validation logits identical before/after"
    ),
}

with open(PROTOCOL, "w") as f:
    json.dump(
        protocol,
        f,
        indent=2,
    )

print("✅ Protocol integrity record updated")

print("\n" + "=" * 105)

if len(bad) == 0:
    print("✅ FINAL FLOODSHIFT CHECKPOINT IS FULLY FINITE AND ARCHIVE-READY")
else:
    print("❌ STOP — unexpected non-finite parameter remains")

print("=" * 105)

FINAL EXPORT SANITATION
Backup created:
  /kaggle/working/FloodShift_FINAL_TerraMind_RIFT/checkpoints/FloodShift_FINAL_TerraMind_RIFT_best_ORIGINAL_UNSANITIZED.pt

Before sanitation
Shape       : (768,)
Finite      : False
NaNs        : 85
Infs        : 0

✅ Final checkpoint sanitized

Non-finite tensors: []
NEW SHA256: 6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822
✅ Protocol integrity record updated

✅ FINAL FLOODSHIFT CHECKPOINT IS FULLY FINITE AND ARCHIVE-READY


In [31]:
import os
from IPython.display import FileLink, display

ZIP_PATH = "/kaggle/working/FloodShift_FINAL_ARCHIVE.zip"

assert os.path.exists(ZIP_PATH), "ZIP not found!"
print(
    "ZIP exists ✅",
    f"{os.path.getsize(ZIP_PATH)/(1024**2):.2f} MB"
)

# Important: serve it as a relative file from /kaggle/working
os.chdir("/kaggle/working")

display(
    FileLink(
        "FloodShift_FINAL_ARCHIVE.zip",
        result_html_prefix="<h3>⬇️ CLICK HERE TO DOWNLOAD:</h3>"
    )
)

ZIP exists ✅ 408.69 MB


/kaggle/working/FloodShift_FINAL_ARCHIVE.zip